<style>
.topic-header { background: linear-gradient(135deg, #e8f4f8 0%, #d4e8f0 100%); border-left: 4px solid #5ba4c9; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0; font-size: 15px; color: #1a3a4a; }
.concept-box { background: #eef6fa; border: 1px solid #c4dce8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a4a5a; }
.try-it { background: #fef9e7; border: 1px solid #f0d87a; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #5a4a1a; }
.takeaway { background: #e8f5e8; border: 1px solid #a8d5a8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a5a2a; }
.warning-box { background: #fdf0f0; border: 1px solid #e8b0b0; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #6a2a2a; }
.where-box { background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a3000; }
.fix-box { background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #1b5e20; }
.diagram-box { background: #f8f9fa; border: 2px solid #dee2e6; border-radius: 12px; padding: 24px; margin: 16px 0; text-align: center; }
.compare-table { width: 100%; border-collapse: collapse; margin: 12px 0; }
.compare-table th { background: #d4e8f0; color: #1a3a4a; padding: 10px 14px; text-align: left; border: 1px solid #c4dce8; }
.compare-table td { padding: 10px 14px; border: 1px solid #dee2e6; font-size: 13px; }
.compare-table tr:nth-child(even) { background: #f8fbfd; }
.section-divider { border: none; border-top: 2px solid #d4e8f0; margin: 25px 0; }
</style>

<div class='topic-header'>
<h1>E08 &middot; Day 3 &middot; RAG End-to-End &mdash; Ground the Model in YOUR Documents</h1>
<p><strong>GenAI for Engineering Managers &bull; Exercise 8 of 15 &bull; Opens Day 3</strong> &nbsp;|&nbsp; From an assistant that chats fluently to one that answers from your handbook and runbooks &mdash; with sources</p>
</div>

**Why this matters at your altitude.** Every "AI assistant for our docs" pitch your teams will bring you &mdash; support copilots, onboarding bots, runbook assistants &mdash; is, underneath, the pattern in this notebook: **Retrieval-Augmented Generation (RAG)**. In the next hour you will watch a raw model fail on questions about our own engineering handbook, then watch the *same model* answer them correctly &mdash; with citations &mdash; after we give it a retrieval layer. No fine-tuning, no training run, no data leaving the room. When a vendor quotes months for "training the AI on your documents", this session is your calibration for what that sentence should actually mean, cost, and take.

<div style="background: linear-gradient(135deg, #e8f4f8 0%, #c4dce8 100%); padding: 30px 32px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: #1a3a4a; margin: 0; font-size: 28px;">A2 &middot; Day 4 &middot; Memory and Perception</h1>
<p style="color: #3a6a8a; margin: 8px 0 0 0; font-size: 16px;">GenAI for Engineering Managers &mdash; Exercise 2 of 3 &middot; Facilitator-run (watch, or try alongside)</p>
<p style="color: #2a4a5a; margin: 14px 0 0 0; font-size: 14px; line-height: 1.6;">
<strong>The gap A1 left:</strong> we fixed the assistant's amnesia with a Python list that grows on every
turn and vanishes when the kernel restarts. That is not memory &mdash; it is a variable.
This session makes memory <strong>bounded</strong> and makes it <strong>survive a restart</strong>,
then turns the agent the other way round: not just remembering what happened <em>inside</em> the
conversation, but looking <strong>outward</strong> at a world that moved on without it.
</p>
</div>

<div class='concept-box'>
<strong>Two directions, one idea.</strong> Memory is the agent looking <em>inward</em> at what it has already
been told. Search is the agent looking <em>outward</em> at what it was never told. Both exist because a
model on its own knows only what was frozen into it at training time.
</div>

<hr class='section-divider'>

## Part 1 &mdash; Setup

In [ ]:
%pip install -qU openai ddgs

In [ ]:
import os, json, datetime, subprocess, sys, textwrap

os.environ['OPENAI_API_KEY'] = 'PASTE_THE_KEY_SHARED_IN_SESSION_HERE'

from openai import OpenAI
client = OpenAI()
MODEL  = 'gpt-4.1-nano'

def chat(messages, tools=None):
    """One call to the model. Returns the message object."""
    kwargs = dict(model=MODEL, messages=messages)
    if tools:
        kwargs['tools'] = tools
    return client.chat.completions.create(**kwargs).choices[0].message

print('Ready:', MODEL)

### The conversation we will keep losing

<div class='concept-box'>
A realistic on-call exchange about the POS sync incident from A1. Eight turns. The facts that matter
&mdash; <strong>Store 4471</strong> and a <strong>backlog of 900</strong> &mdash; are stated early and never
repeated. That is exactly how real conversations work, and exactly why memory is hard.
</div>

In [ ]:
CONVERSATION = [
    ('user',      "We've got POS_SYNC_LAG firing. Ticket is INC-30021."),
    ('assistant', "Understood. What's the backlog size and which store?"),
    ('user',      "Store 4471, Bentonville. Backlog crossed 900 and it's climbing about 120 a minute."),
    ('assistant', "That's above the 750 warning threshold but below the 2,000 critical mark."),
    ('user',      "Edge gateway heartbeat looks fine and no other store in the region is affected."),
    ('assistant', "Isolated store with a healthy gateway — likely a stuck consumer group."),
    ('user',      "Right. I'll watch it for ten minutes."),
    ('assistant', "Sensible. Escalate if it crosses 2,000."),
]

QUESTION = 'Which store was that, and what was the backlog when we last spoke?'

for i, (role, text) in enumerate(CONVERSATION, 1):
    print(f'{i}. {role:>9}: {text}')
print(f'\nThe question we will ask after all this: "{QUESTION}"')
print('The answer lives in turn 3 — five turns back.')

<hr class='section-divider'>

## Part 2 &mdash; The Audit: Measure the Gap Before Fixing It

<div class='concept-box'>
Two runs. Once with nothing, once with everything. These are the floor and the ceiling &mdash; every
memory strategy in this notebook lands somewhere between them.
</div>

In [ ]:
print('A. STATELESS — the model gets only the question')
print('-' * 72)
print(chat([{'role': 'user', 'content': QUESTION}]).content.strip())

print('\nB. FULL HISTORY — the model gets all 8 turns')
print('-' * 72)
msgs = [{'role': r, 'content': c} for r, c in CONVERSATION] + [{'role': 'user', 'content': QUESTION}]
print(chat(msgs).content.strip())

<div class='takeaway'>
<strong>The floor and the ceiling.</strong> Stateless cannot answer &mdash; not because the model is weak,
but because nobody told it. Full history answers precisely.<br><br>
So why not always send everything? Because "everything" is unbounded, and you pay for every token of
it on <em>every single turn</em>. That is the trade this whole session is about.
</div>

<hr class='section-divider'>

## Part 3 &mdash; Type 1 &middot; KEEP: a window of recent turns

<div class='concept-box'>
The simplest bounded memory: keep only the last <em>k</em> turns and throw the rest away. Cheap,
predictable, and it works right up until the fact you need falls off the back of the window.<br><br>
Rather than guess at <em>k</em>, we <strong>measure it</strong> &mdash; and we check the answer for the two
facts that matter instead of eyeballing whether it "looks right".
</div>

In [ ]:
print(f'Question: {QUESTION}')
print('Looking for: store "4471" and backlog "900"')
print('=' * 78)

for k in (2, 4, 6, 8):
    window = CONVERSATION[-k:]
    msgs   = [{'role': r, 'content': c} for r, c in window] + [{'role': 'user', 'content': QUESTION}]
    answer = chat(msgs).content.strip()

    got_store   = '4471' in answer
    got_backlog = '900'  in answer
    verdict     = 'CORRECT' if (got_store and got_backlog) else 'FAILED '

    print(f'k={k:>2} turns | {verdict} | store 4471: {"Y" if got_store else "n"} '
          f'| backlog 900: {"Y" if got_backlog else "n"}')
    print(f'         {answer[:110]}')
    print('-' * 78)

<div class='takeaway'>
<strong>There is the threshold, measured rather than guessed.</strong> The window has to reach back far
enough to include turn 3. Anything shorter drops the only turn that contained the answer &mdash; and
note <em>how</em> it fails: not with an error, but with a polite "I don't have access to that".<br><br>
<strong>The management point:</strong> a windowed agent does not tell you when it has forgotten something.
It degrades silently, and it degrades <em>as conversations get longer</em> &mdash; which means it will look
perfect in a demo and fail with your busiest customers.
</div>

### What the window costs

In [ ]:
def rough_tokens(messages):
    """Crude but honest: ~4 characters per token."""
    return sum(len(m['content']) for m in messages) // 4

print('Tokens resent on EVERY turn, by strategy:')
print('-' * 56)
for label, msgs in [
    ('stateless',        [{'role': 'user', 'content': QUESTION}]),
    ('window k=4',       [{'role': r, 'content': c} for r, c in CONVERSATION[-4:]] + [{'role':'user','content':QUESTION}]),
    ('window k=6',       [{'role': r, 'content': c} for r, c in CONVERSATION[-6:]] + [{'role':'user','content':QUESTION}]),
    ('full history (8)', [{'role': r, 'content': c} for r, c in CONVERSATION]      + [{'role':'user','content':QUESTION}]),
]:
    print(f'  {label:<18} ~{rough_tokens(msgs):>4} tokens')

print('\nThis conversation is 8 turns. A real support thread runs to hundreds,')
print('and "full history" grows without limit — every turn, forever.')

<div class='concept-box'>
<strong>Two more strategies exist, and you should know their names even though we are not building them
today:</strong><br><br>
<strong>COMPRESS</strong> &mdash; summarise old turns into a paragraph and keep recent turns verbatim.
Bounded like a window, but keeps the gist of what fell off. You lose exact figures.<br>
<strong>RETRIEVE</strong> &mdash; embed every past turn and retrieve only the relevant ones. This is RAG
pointed at the conversation instead of at documents. Best recall, most machinery.
</div>

<hr class='section-divider'>

## Part 4 &mdash; Type 2 &middot; PERSIST: memory that survives a restart

<div class='concept-box'>
Every strategy so far lives in a Python variable. Restart the kernel, redeploy the service, scale to a
second server &mdash; and the customer starts from zero.<br><br>
We are not going to <em>describe</em> this. We will write the memory to a file, then start a
<strong>genuinely separate Python process</strong> that has never seen this notebook's variables, and
watch what it can and cannot do.
</div>

In [ ]:
MEMORY_FILE = 'agent_memory.json'

with open(MEMORY_FILE, 'w') as f:
    json.dump([{'role': r, 'content': c} for r, c in CONVERSATION], f, indent=2)

print(f'Wrote {MEMORY_FILE}:')
print(open(MEMORY_FILE).read()[:340], '...')

In [ ]:
# A standalone script. It shares NO variables with this notebook — it only has the file.
worker = '''
import json, os, sys
from openai import OpenAI

QUESTION = sys.argv[1]
client   = OpenAI()

messages = []
if os.path.exists("agent_memory.json"):
    messages = json.load(open("agent_memory.json"))
    print(f"[new process] loaded {len(messages)} turns from disk")
else:
    print("[new process] no memory file found — starting from nothing")

messages.append({"role": "user", "content": QUESTION})
reply = client.chat.completions.create(model="gpt-4.1-nano", messages=messages)
print("ANSWER:", reply.choices[0].message.content.strip())
'''
open('memory_worker.py', 'w').write(worker)
print('Wrote memory_worker.py — a separate program.')

In [ ]:
def run_worker():
    result = subprocess.run([sys.executable, 'memory_worker.py', QUESTION],
                            capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()

print('RUN 1 — fresh process, memory file present')
print('=' * 72)
print(run_worker())

os.rename(MEMORY_FILE, '_hidden_memory.json')          # the control: same code, no file

print('\nRUN 2 — fresh process, memory file hidden (the control)')
print('=' * 72)
print(run_worker())

os.rename('_hidden_memory.json', MEMORY_FILE)

<div class='takeaway'>
<strong>Same program, same question, same model &mdash; run twice.</strong> The only difference is whether a
file existed on disk. That is the whole of persistent memory: <strong>state that outlives the process
that created it.</strong><br><br>
Notice this is not an AI technique at all. It is a database decision wearing a new hat. Which is
precisely why it belongs in your architecture review and not in a prompt-engineering discussion.
</div>

<div class='warning-box'>
<strong>And the question a manager must ask next:</strong> that file now contains a customer's
conversation. Where does it live, who can read it, how long is it kept, and what happens when the
customer asks you to delete it? <strong>Persistent memory turns a stateless feature into a data-retention
obligation</strong>, and that conversation belongs with legal before launch, not after.
</div>

<hr class='section-divider'>

## Part 5 &mdash; The Other Direction: Perception

<div class='concept-box'>
Memory looks <em>inward</em>. But the agent has a second, larger blind spot: <strong>the world moved on
after its training data ended.</strong> No amount of memory fixes that &mdash; you cannot remember what
you were never told.<br><br>
Watch the model answer a question about current events with no tool at all. Read the first few words
of its reply especially carefully.
</div>

In [ ]:
CURRENT_Q = "What are the most recent developments in Walmart's supply chain automation?"

print('WITHOUT any tool:')
print('=' * 72)
print(chat([{'role': 'user', 'content': CURRENT_Q}]).content.strip()[:500])

<div class='warning-box'>
<strong>Look at how it opened.</strong> It dated its own answer to its training cutoff &mdash; and then
answered anyway, fluently, as though that were good enough. It is not lying; it genuinely has nothing
newer. But an executive skim-reading that paragraph would take it as current.
</div>

### Give it eyes: a web search tool

<div class='concept-box'>
Exactly the same tool mechanism as A1 &mdash; a Python function, described in English, that the model may
choose to call. This one reaches the public web through DuckDuckGo.<br><br>
<strong>Note the error handling.</strong> Free search endpoints rate-limit, and a classroom of twenty
people hitting one at once is exactly when it happens. The tool degrades to a clear message instead of
a traceback, so the lesson survives a bad network.
</div>

In [ ]:
def web_search(query, max_results=4):
    """Search the public web. Degrades gracefully if the endpoint is unavailable."""
    try:
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=max_results))
        return {'status': 'live',
                'results': [{'title': h.get('title'),
                             'url': h.get('href'),
                             'snippet': (h.get('body') or '')[:220]} for h in hits]}
    except Exception as exc:
        return {'status': 'unavailable',
                'note': f'Search endpoint unavailable ({type(exc).__name__}). '
                        f'Tell the user you could not verify current information.',
                'results': []}

SEARCH_TOOL = [{
    'type': 'function',
    'function': {
        'name': 'web_search',
        'description': 'Search the public web for current information that is more recent than '
                       'your training data. Use for any question about recent or current events.',
        'parameters': {'type': 'object',
                       'properties': {'query': {'type': 'string'}},
                       'required': ['query']},
    },
}]

probe = web_search('Walmart supply chain automation')
print('Search tool status:', probe['status'], '|', len(probe['results']), 'results')
if probe['results']:
    print('  e.g.', probe['results'][0]['title'][:80])

In [ ]:
def research_agent(question, system, max_steps=5, show_trace=True):
    messages = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': question}]
    queries  = []
    for step in range(max_steps):
        msg = chat(messages, tools=SEARCH_TOOL)
        if not msg.tool_calls:
            return msg.content.strip(), queries
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or '{}')
            queries.append(args.get('query'))
            if show_trace:
                print(f'   [step {step + 1}] searched: "{args.get("query")}"')
            messages.append({'role': 'tool', 'tool_call_id': call.id,
                             'content': json.dumps(web_search(**args))[:3000]})
    return '(too many steps)', queries


print('WITH the web_search tool:')
print('=' * 72)
answer, queries = research_agent(
    CURRENT_Q, 'You are a research assistant. Use web_search for anything current. Cite your sources.')
print('\n' + answer[:700])

<div class='takeaway'>
<strong>Same model, same question, one tool.</strong> It stopped answering from a frozen snapshot and
went and looked &mdash; and came back attributing claims to <strong>named, checkable sources</strong>
rather than to itself. That is the difference between an assistant that <em>recalls</em> and one that
<em>checks</em>.<br><br>
<em>(How much of a citation you get &mdash; a publication name, a date, a full URL &mdash; varies run to
run. If your product depends on clickable citations, that is a requirement to specify and test, not
something to hope the model does.)</em>
</div>

### The trap hiding in that success

<div class='warning-box'>
Read the search query in the trace above. Does it carry <em>this</em> year &mdash; or a stale one, or
no year at all?<br><br>
The model wrote that query itself, using its own frozen sense of "now". We gave it eyes and forgot to
tell it what day it is. The cell below runs both versions and counts the difference rather than
asking you to take it on trust.
</div>

In [ ]:
today = datetime.date.today().isoformat()

print(f"Actual date today: {today}\n")

print('A. plain system prompt')
_, q_plain = research_agent(CURRENT_Q,
    'You are a research assistant. Use web_search for current information.',
    show_trace=False)
print('   queries:', q_plain)

print(f"\nB. today's date injected into the system prompt")
_, q_dated = research_agent(CURRENT_Q,
    f"You are a research assistant. Today's date is {today}. Use web_search for current "
    f"information, and make your search queries reflect today's date.",
    show_trace=False)
print('   queries:', q_dated)

year = today[:4]
print(f"\n  queries mentioning the correct year ({year}):")
print(f"    plain           : {sum(year in (q or '') for q in q_plain)} of {len(q_plain)}")
print(f"    date injected   : {sum(year in (q or '') for q in q_dated)} of {len(q_dated)}")

<div class='takeaway'>
<strong>This is the most useful thing in the notebook, and it is easy to miss.</strong><br><br>
The agent had a search tool and used it correctly &mdash; and still aimed it at the wrong point in
time, because nobody told it what year it was. Its query carried a stale year or none at all, so the
results came back real, relevant, and <strong>anchored to the wrong period</strong>. Nothing errored.
Nothing looked wrong. Note the answer above even dated itself "as of 2024".<br><br>
<strong>Giving an agent a capability is not the same as giving it the context to use that capability
well.</strong> In A1 we saw the model refuse to use knowledge it had; here we see it use a tool
competently toward the wrong target. Both failures are invisible in the output.<br><br>
<strong>The fix cost one sentence in the system prompt.</strong> Ask your teams what else their agents
are assuming.
</div>

<hr class='section-divider'>

## Recap

<table class='compare-table'>
<tr><th>Part</th><th>What we proved</th><th>Manager takeaway</th></tr>
<tr><td>2</td><td>Stateless failed; full history answered exactly</td><td>The floor and the ceiling — everything else is a cost trade</td></tr>
<tr><td>3</td><td>Measured the k at which the window stops working</td><td>Windowed memory degrades <strong>silently</strong>, and worsens as conversations lengthen</td></tr>
<tr><td>3</td><td>Priced each strategy in tokens per turn</td><td>"Send everything" is a bill that grows forever</td></tr>
<tr><td>4</td><td>A separate process answered from a file, and failed without it</td><td>Persistence is a database decision, not a prompt one</td></tr>
<tr><td>4</td><td>&mdash;</td><td>Stored conversations create retention and deletion obligations</td></tr>
<tr><td>5</td><td>No tool: answered from a frozen snapshot, dated to its cutoff</td><td>A confident answer can be years old and never say so</td></tr>
<tr><td>5</td><td>With search: real, citable sources</td><td>The difference between recalling and checking</td></tr>
<tr><td>5</td><td>Searched the wrong year until we injected today's date</td><td><strong>A capability without context fails invisibly</strong></td></tr>
</table>

<div class='concept-box'>
<strong>The four memory types &mdash; know all four, we built two</strong>
<table class='compare-table'>
<tr><th>Type</th><th>What it does</th><th>Costs you</th><th>Built today</th></tr>
<tr><td><strong>KEEP</strong></td><td>Last k turns verbatim</td><td>Silently forgets older facts</td><td>Yes</td></tr>
<tr><td><strong>COMPRESS</strong></td><td>Summarise old, keep recent</td><td>Loses exact figures</td><td>Named only</td></tr>
<tr><td><strong>PERSIST</strong></td><td>Survives restarts and servers</td><td>Storage, privacy, deletion duties</td><td>Yes</td></tr>
<tr><td><strong>RETRIEVE</strong></td><td>Embed turns, fetch relevant ones</td><td>Most machinery; RAG over the chat</td><td>Named only</td></tr>
</table>
</div>

In [ ]:
# Housekeeping — remove the files this notebook wrote.
for path in (MEMORY_FILE, 'memory_worker.py'):
    if os.path.exists(path):
        os.remove(path)
        print('removed', path)
print('Clean.')

<div class='topic-header'>
<strong>&#128279; The gap we leave &mdash; and where Exercise 3 begins</strong><br><br>
Our agent can now <strong>act</strong> (A1), <strong>remember</strong> across restarts, and
<strong>check the outside world</strong>. It is, by any reasonable definition, capable.<br><br>
And nobody has approved a single thing it has done.<br><br>
In A1 Part 6 it changed our system of record because a tool existed and it decided to use it. Add
persistent memory and web access to that, and the honest question is no longer "can it?" but
<strong>"who says yes?"</strong><br><br>
Exercise 3 puts a human in front of the action &mdash; a real <strong>Approve / Reject</strong> button on a
real interface &mdash; and proves that a rejected action leaves the system of record untouched.
</div>